# Fatigue modeling

Ordinal models for `fatigue_num` (0–5) with participant-level held-out test, GroupKFold CV, and Optuna tuning. Core logic lives in `src/modeling/`.

Tuning and CV use **train/val participants only**; held-out test participants never appear in Optuna or CV folds.

In [2]:
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sys
from pathlib import Path

_src = Path('../../src').resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# Ensure local src edits are picked up when re-running this cell.
for _mod in [k for k in list(sys.modules) if k == 'modeling' or k.startswith('modeling.')]:
    del sys.modules[_mod]

import pandas as pd
from modeling.baselines import (
    run_all_baseline_benchmarks,
    summarize_baseline_metrics,
)
from modeling.config import (
    DATA_PATH,
    N_CV_FOLDS,
    OPTUNA_TRIALS,
)
from modeling.data import load_fatigue_data, prepare_splits, split_summary_table
from modeling.registry import ORDINAL_MODELS
from modeling.runner import tune_and_benchmark_model
from modeling.summaries import build_history_ablation_summary, collect_summaries


## 1. Load data and split

Participants are held out with a **stratified split** on per-participant **mean fatigue** (`fatigue_num` averaged over each participant's days; `prepare_splits(..., stratify=True)`) so train/val and test have similar average fatigue levels.

“We randomly assign whole participants to train/val or test, but we do it in a way that both groups contain a similar mix of people with low, medium, and high average fatigue — not just a random 8 people who might all happen to be high-average-fatigue reporters.”


In [4]:
df = load_fatigue_data('../../' + DATA_PATH)
bundle = prepare_splits(df)

print(f"Rows: {len(df):,}  Participants: {df['id'].nunique()}")
display(split_summary_table(bundle))
print('Test participant ids:', sorted(bundle.test_ids))


Rows: 3,331  Participants: 42


,split,participants,rows,mean_fatigue
0,train_val,34,2659,2.462204
1,test,8,672,2.653274


Test participant ids: [np.int64(7), np.int64(14), np.int64(24), np.int64(38), np.int64(40), np.int64(41), np.int64(46), np.int64(50)]


Re-run the **init accumulators** cell below before a fresh partial run to clear prior tuned-model results.


In [5]:
# Re-run this cell to clear accumulated model results before a fresh partial run.
ordinal_results = []
history_ordinal_results = []

ordinal_best_params = {}
history_best_params = {}


## 2. Baseline benchmarks

Simple predictors evaluated with the same GroupKFold CV and held-out test protocol as the tuned models. Includes persistence baselines **`lag1_fatigue`** and **`expanding_mean`**.


In [6]:
ordinal_baseline_results = run_all_baseline_benchmarks(bundle, n_splits=N_CV_FOLDS)

ordinal_baseline_summary = summarize_baseline_metrics(ordinal_baseline_results)

print('Ordinal baselines (test metrics)')
display(ordinal_baseline_summary[[c for c in ordinal_baseline_summary.columns if c.startswith('test_')]])


Ordinal baselines (test metrics)


,test_mae,test_rmse,test_r2,test_qwk
model,,,,
global_mean,1.406250,1.640721,-0.188402,0.000000
global_mode,1.156250,1.544479,-0.053072,0.000000
lag1_fatigue,0.950893,1.424175,0.104593,0.549449
expanding_mean,1.025298,1.336863,0.211017,0.422289


MAE: Mean Absolute Error;

RMSE: Root Mean Squared Error, measures the variation in residual/error

R2: how much variability is explained by the model

QWK: Quadratic Weighted Kappa. QWK measures the agreement between two raters—such as an AI and a human—on an ordered scale. It is designed to adjust for chance agreements and heavily penalize larger scoring discrepancies over minor ones.

## 3. Train/Tune models

### Ordinal Regression

Continuous loss on `fatigue_num`, then round and clip to [0, 5].

#### `linear_regression`


In [ ]:
_name = 'linear_regression'
_result, _params = tune_and_benchmark_model(
    # feature_set defaults to 'base' (17 daily features only)
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] linear_regression  test_mae=1.3452


#### `ordinal_rf`


In [ ]:
_name = 'ordinal_rf'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_rf  test_mae=1.4062


#### `catboost_regressor`


In [ ]:
_name = 'catboost_regressor'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_regressor  test_mae=1.2381


### Ordinal Classification

Ordered likelihood or threshold structure on `fatigue_num` 0–5. Evaluated with the same MAE / QWK metrics as regression models.

#### `ordered_logistic`


In [ ]:
_name = 'ordered_logistic'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordered_logistic  test_mae=1.3095


#### `ordinal_forest`


In [ ]:
_name = 'ordinal_forest'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_forest  test_mae=1.2202


#### `population_ordered_logistic`


This model does not assume a different baseline for each participant -- this is because we want the model to generalize to the population.

In training, this model only uses day-varying features and deliberately drops participant-level constants such as age, age_of_first_menarche, etc. The reason is that the model does not want to rely on participant-specific demographics.

In [ ]:
_name = 'population_ordered_logistic'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] population_ordered_logistic  test_mae=1.4554


#### `catboost_ordinal`


In [ ]:
_name = 'catboost_ordinal'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_ordinal  test_mae=1.1518


### History (feature ablation)

Same seven ordinal models as above, with **seven extra leakage-safe history columns** appended to the daily feature matrix. History features use fixed defaults from `prepare_splits` (`ewma_alpha=0.3`, `rolling_window=3`); first-day NaNs in history columns are imputed with the train/val median.

**History features** (7 cols):
- fatigue lag1: Yesterday's fatigue score
- fatigue EWMA: Exponentially weighted average of past fatigue; recent days count more
- fatigue expanding mean: Average fatigue on all earlier days for this person
- fatigue delta lag1: Change in fatigue, the worsening/improving trend
- activity_logsum_roll3_mean: Rolling mean of prior days' sum of log1p(lightly) + log1p(moderately) + log1p(very)
- calories_sum_roll3_mean: Recent typical daily calories burned
- very_roll3_mean: Recent typical "very active" minutes


#### Ordinal Regression (history)

Continuous loss on `fatigue_num`, then round and clip to [0, 5].


##### `linear_regression` (history)


In [ ]:
_name = 'linear_regression'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] linear_regression_history  test_mae=0.8899


##### `ordinal_rf` (history)


In [ ]:
_name = 'ordinal_rf'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_rf_history  test_mae=0.8884


##### `catboost_regressor` (history)


In [ ]:
_name = 'catboost_regressor'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_regressor_history  test_mae=0.9018


#### Ordinal Classification (history)


##### `ordered_logistic` (history)


In [ ]:
_name = 'ordered_logistic'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordered_logistic_history  test_mae=0.8929


##### `ordinal_forest` (history)


In [ ]:
_name = 'ordinal_forest'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_forest_history  test_mae=0.8958


##### `population_ordered_logistic` (history)


In [ ]:
_name = 'population_ordered_logistic'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] population_ordered_logistic_history  test_mae=0.8720


##### `catboost_ordinal` (history)


In [ ]:
_name = 'catboost_ordinal'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    feature_set='history',
    display_name=f'{_name}_history',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[f'{_name}_history'] = _params
print(f'[ok] {_name}_history  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_ordinal_history  test_mae=0.8884


## 4. Results summary

Aggregates §2 baselines plus any §3 models run (base and/or `_history` variants). Uses `collect_summaries` from `modeling.summaries`:

- **CV table:** mean metrics over GroupKFold folds on train/val participants (`cv_mae_std` = fold-to-fold MAE stability).
- **Test table:** one evaluation per model after refitting on **all** train/val rows with Optuna `best_params`, scored on held-out test participants.

The next cell walks through the aggregation; the cell after that compares base vs history test MAE.

In [28]:
# Merge baselines (§2), base tuned models (§3), and history variants (§3 History).
# globals().get(...) allows partial notebook runs without NameError on skipped cells.

ordinal_results = globals().get('ordinal_results', [])
history_ordinal_results = globals().get('history_ordinal_results', [])
ordinal_best_params = globals().get('ordinal_best_params', {})
history_best_params = globals().get('history_best_params', {})

ran_tuned_models = sorted(set(ordinal_best_params) | set(history_best_params))
print(f'Ran {len(ran_tuned_models)} tuned ordinal models: {ran_tuned_models}')

all_ordinal_results = ordinal_baseline_results + ordinal_results + history_ordinal_results

ordinal_cv_summary, ordinal_test_summary = collect_summaries(all_ordinal_results)

print('CV summary (baselines first; cv_* = mean over GroupKFold folds on train/val)')
display(ordinal_cv_summary)
print('Held-out test summary (model refit on full train/val, scored on test participants)')
display(ordinal_test_summary)


Ran 14 tuned ordinal models: ['catboost_ordinal', 'catboost_ordinal_history', 'catboost_regressor', 'catboost_regressor_history', 'linear_regression', 'linear_regression_history', 'ordered_logistic', 'ordered_logistic_history', 'ordinal_forest', 'ordinal_forest_history', 'ordinal_rf', 'ordinal_rf_history', 'population_ordered_logistic', 'population_ordered_logistic_history']
CV summary (baselines first; cv_* = mean over GroupKFold folds on train/val)


,best_params,cv_mae,cv_rmse,cv_r2,cv_qwk,cv_mae_std
model,,,,,,
global_mean,{},1.348516,1.606173,-0.297646,0.000000,0.153930
global_mode,{},1.216046,1.554387,-0.200378,0.000000,0.243084
lag1_fatigue,{},0.824015,1.315205,0.096947,0.546287,0.193110
expanding_mean,{},0.867974,1.198179,0.280799,0.495990,0.127344
linear_regression,{'alpha': 8.959365126002776},1.465313,1.731319,-0.518947,-0.004255,0.239195
ordinal_rf,"{'n_estimators': 141, 'max_depth': 17, 'min_sa...",1.285987,1.597586,-0.266228,0.070166,0.170034
catboost_regressor,"{'iterations': 435, 'depth': 10, 'learning_rat...",1.248556,1.521498,-0.156448,0.091178,0.138112
ordered_logistic,{'alpha': 9.744359226057488},1.546936,1.826368,-0.717851,-0.014253,0.213686
ordinal_forest,"{'n_estimators': 336, 'max_depth': 12, 'min_sa...",1.227035,1.494982,-0.117982,0.124237,0.125846


Held-out test summary (model refit on full train/val, scored on test participants)


,best_params,test_mae,test_rmse,test_r2,test_qwk
model,,,,,
global_mean,{},1.406250,1.640721,-0.188402,0.000000
global_mode,{},1.156250,1.544479,-0.053072,0.000000
lag1_fatigue,{},0.950893,1.424175,0.104593,0.549449
expanding_mean,{},1.025298,1.336863,0.211017,0.422289
linear_regression,{'alpha': 8.959365126002776},1.345238,1.622021,-0.161467,-0.004347
ordinal_rf,"{'n_estimators': 141, 'max_depth': 17, 'min_sa...",1.406250,1.729902,-0.321103,-0.044814
catboost_regressor,"{'iterations': 435, 'depth': 10, 'learning_rat...",1.238095,1.554563,-0.066868,0.057303
ordered_logistic,{'alpha': 9.744359226057488},1.309524,1.642987,-0.191686,-0.005966
ordinal_forest,"{'n_estimators': 336, 'max_depth': 12, 'min_sa...",1.220238,1.542069,-0.049788,0.056225


In [29]:
# --- Base vs history ablation (test MAE only) ---
# delta_mae = history - base; negative means history features improved test MAE.

history_ablation_summary = build_history_ablation_summary(ordinal_test_summary, ORDINAL_MODELS)
if history_ablation_summary.empty:
    print('No paired base/history models found — run both §3 blocks first.')
else:
    print('Base vs history paired comparison (delta_mae = history - base; negative = history helps)')
    display(history_ablation_summary)


Base vs history paired comparison (delta_mae = history - base; negative = history helps)


,test_mae_tabular,test_mae_history,delta_mae
model,,,
population_ordered_logistic,1.455357,0.872024,-0.583333
ordinal_rf,1.406250,0.888393,-0.517857
linear_regression,1.345238,0.889881,-0.455357
ordered_logistic,1.309524,0.892857,-0.416667
catboost_regressor,1.238095,0.901786,-0.336310
ordinal_forest,1.220238,0.895833,-0.324405
catboost_ordinal,1.151786,0.888393,-0.263393
